In [1]:
import pandas as pd
import numpy as np

facebook_ads = pd.read_csv('../data/marketing/Facebook Ads.csv')
facebook_ads.head()

,Ad name,Delivery status,Delivery level,Reach,Impressions,Frequency,Result type,Results,Amount spent (THB),Cost per result,Starts,Ends,Rest ID,Rest 2,Rest 3,Rest 4,Rest 5
0,SRC=KOL|COUNTRY=TH|RID=4503+837|RNAME=COPPER_B...,active,ad,156467,332013,2.121936,Website purchases,689,7396.81,10.735573,20/03/2025,NaN,4503,837.0,NaN,NaN,NaN
1,SRC=KOL|COUNTRY=TH|RID=4503+837|RNAME=COPPER_B...,active,ad,78081,167712,2.147923,Website purchases,340,5438.39,15.995265,20/03/2025,NaN,4503,837.0,NaN,NaN,NaN
2,SRC=KOL|COUNTRY=TH|RID=4503+837|RNAME=COPPER_B...,active,ad,56111,85464,1.523124,Website purchases,191,2230.25,11.676702,20/03/2025,NaN,4503,837.0,NaN,NaN,NaN
3,SRC=KOL|COUNTRY=TH|RID=4503+837|RNAME=COPPER_B...,active,ad,37510,66331,1.768355,Website purchases,117,2158.46,18.448376,20/03/2025,NaN,4503,837.0,NaN,NaN,NaN
4,SRC=KOL|COUNTRY=TH|RID=4503+837|RNAME=COPPER_B...,active,ad,4327,5107,1.180263,Website purchases,3,224.04,74.680000,20/03/2025,NaN,4503,837.0,NaN,NaN,NaN


In [2]:
facebook_ads.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3987 entries, 0 to 3986
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Ad name             3987 non-null   object 
 1   Delivery status     3987 non-null   object 
 2   Delivery level      3987 non-null   object 
 3   Reach               3987 non-null   int64  
 4   Impressions         3987 non-null   int64  
 5   Frequency           3987 non-null   float64
 6   Result type         3191 non-null   object 
 7   Results             3182 non-null   object 
 8   Amount spent (THB)  3987 non-null   float64
 9   Cost per result     3191 non-null   float64
 10  Starts              3987 non-null   object 
 11  Ends                1816 non-null   object 
 12  Rest ID             3432 non-null   object 
 13  Rest 2              43 non-null     float64
 14  Rest 3              6 non-null      float64
 15  Rest 4              1 non-null      float64
 16  Rest 5

In [3]:
def parse_ad_name_intelligent(ad_name):
    """
    Parse ad name handling two formats:
    1. Structured: SRC=KOL|COUNTRY=TH|RID=4503+837|RNAME=COPPER_BEYOND_BUFFET
    2. Legacy: kol-pfd_hh-mp_brn_393_zaap-classic
    """
    parsed = {
        'SRC': None,
        'COUNTRY': None,
        'RID': None,
        'RNAME': None,
        'TYPE': None,
        'CAMP': None,
        'KOLID': None,
        'PLAT': None,
        'DATE': None,
        'Format_Type': None
    }
    
    if pd.isna(ad_name) or not isinstance(ad_name, str):
        return parsed
    
    # Check if it's the structured Key=Value format
    if '|' in ad_name and '=' in ad_name:
        parsed['Format_Type'] = 'STRUCTURED'
        parts = ad_name.split('|')
        for part in parts:
            if '=' in part:
                key, value = part.split('=', 1)
                parsed[key.strip()] = value.strip()
    
    # Check if it's the legacy underscore-separated format
    elif '_' in ad_name:
        parsed['Format_Type'] = 'LEGACY'
        
        # Legacy format pattern analysis:
        # Example 1: kol-pfd_hh-mp_brn_393_zaap-classic
        #   - Segment 0: kol-pfd (KOL identifier, ignore this)
        #   - Segment 1: hh-mp (SRC=hh, KOLID=mp or campaign info)
        #   - Segment 2+: Restaurant info
        # Example 2: album_hh_res_grp_top-buffet-2024
        #   - Segment 0: album (campaign type)
        #   - Segment 1: hh (SRC)
        #   - Segment 2+: Restaurant/campaign info
        
        parts = ad_name.split('_')
        
        if len(parts) >= 2:
            # Look for SRC in the segments (hh, mp, etc.)
            src_found = False
            
            for i, part in enumerate(parts[:3]):  # Check first 3 segments for SRC
                part_lower = part.lower()
                
                # Check if this segment contains SRC indicators
                if part_lower.startswith('hh'):
                    parsed['SRC'] = 'HH'
                    src_found = True
                    
                    # Extract KOLID from this segment if it has a hyphen
                    # Example: hh-mp, hh-cth, hh-pls -> KOLID = mp, cth, pls
                    if '-' in part and len(part.split('-')) > 1:
                        kol_parts = part.split('-')
                        parsed['KOLID'] = '-'.join(kol_parts[1:])  # Everything after 'hh-'
                    
                elif part_lower.startswith('mp'):
                    parsed['SRC'] = 'MP'
                    src_found = True
                    
                    # Extract KOLID if present
                    if '-' in part and len(part.split('-')) > 1:
                        kol_parts = part.split('-')
                        parsed['KOLID'] = '-'.join(kol_parts[1:])
                    
                elif part_lower in ['album', 'single', 'crs']:
                    # These are campaign types, not KOL IDs
                    parsed['CAMP'] = part_lower
                    
            # If SRC not found in early segments, mark as UNKNOWN
            if not src_found:
                parsed['SRC'] = 'UNKNOWN'
            
            # Extract campaign info from middle segments
            # Look for segments like 'res', 'grp', 'ctg', etc.
            campaign_parts = []
            restaurant_start_idx = None
            
            for i, part in enumerate(parts):
                part_lower = part.lower()
                
                # Campaign indicators
                if part_lower in ['res', 'grp', 'ctg', 'top-buffet', 'top-rooftop', 
                                  'waterfront', 'hangout', 'dinner', 'corporate']:
                    campaign_parts.append(part)
                
                # Restaurant ID - numeric segment
                elif part.isdigit() and len(part) >= 2:
                    parsed['RID'] = part
                    restaurant_start_idx = i
                    # Everything after the ID is the restaurant name
                    if i + 1 < len(parts):
                        parsed['RNAME'] = '_'.join(parts[i+1:])
                    break
            
            # Build campaign name from identified parts
            if campaign_parts:
                if parsed['CAMP']:
                    parsed['CAMP'] = f"{parsed['CAMP']}_{'_'.join(campaign_parts)}"
                else:
                    parsed['CAMP'] = '_'.join(campaign_parts)
            
            # If no restaurant ID found but we have parts beyond SRC, use them as restaurant name
            if not parsed['RNAME'] and len(parts) > 2:
                # Find where SRC segment is
                src_idx = -1
                for i, part in enumerate(parts[:3]):
                    if part.lower().startswith('hh') or part.lower().startswith('mp'):
                        src_idx = i
                        break
                
                if src_idx >= 0 and src_idx + 1 < len(parts):
                    # Everything after SRC is restaurant/campaign info
                    remaining = parts[src_idx + 1:]
                    if remaining:
                        parsed['RNAME'] = '_'.join(remaining)
                else:
                    # Use everything after first segment
                    parsed['RNAME'] = '_'.join(parts[1:])
    
    else:
        # Simple format - just store as campaign name
        parsed['Format_Type'] = 'SIMPLE'
        parsed['CAMP'] = ad_name
    
    return parsed

# Apply the intelligent parsing
print("\nParsing ad names...")
extracted_features = facebook_ads['Ad name'].apply(parse_ad_name_intelligent).apply(pd.Series)
facebook_ads = pd.concat([facebook_ads, extracted_features], axis=1)


Parsing ad names...


In [4]:
facebook_ads.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3987 entries, 0 to 3986
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Ad name             3987 non-null   object 
 1   Delivery status     3987 non-null   object 
 2   Delivery level      3987 non-null   object 
 3   Reach               3987 non-null   int64  
 4   Impressions         3987 non-null   int64  
 5   Frequency           3987 non-null   float64
 6   Result type         3191 non-null   object 
 7   Results             3182 non-null   object 
 8   Amount spent (THB)  3987 non-null   float64
 9   Cost per result     3191 non-null   float64
 10  Starts              3987 non-null   object 
 11  Ends                1816 non-null   object 
 12  Rest ID             3432 non-null   object 
 13  Rest 2              43 non-null     float64
 14  Rest 3              6 non-null      float64
 15  Rest 4              1 non-null      float64
 16  Rest 5

In [6]:
# Clean Restaurant ID column - remove invalid values
facebook_ads = facebook_ads[
    ~facebook_ads['Rest ID '].isin(['~', '#VALUE!', 'nan', 'UNKNOWN'])
]

In [7]:
# Create a dictionary with the old names as keys and the new names as values
rename_mapping = {
    'SRC': 'Source',
    'RID': 'Restaurant_ID_frm_adtext',
    'RNAME': 'Restaurant Name',
    'CAMP': 'Campaign',
    'KOLID': 'KOL_ID',
    'PLAT': 'Platform',
    'COUNTRY' : 'Country',
    'Rest ID ' : "Restaurant ID",
    'TYPE' : 'Type',

}

facebook_ads = facebook_ads.rename(columns=rename_mapping)
facebook_ads.columns

Index(['Ad name', 'Delivery status', 'Delivery level', 'Reach', 'Impressions',
       'Frequency', 'Result type', 'Results', 'Amount spent (THB)',
       'Cost per result', 'Starts', 'Ends', 'Restaurant ID', 'Rest 2',
       'Rest 3', 'Rest 4', 'Rest 5', 'Source', 'Country',
       'Restaurant_ID_frm_adtext', 'Restaurant Name', 'Type', 'Campaign',
       'KOL_ID', 'Platform', 'DATE', 'Format_Type'],
      dtype='object')

In [8]:
# 3. Process Timestamps
# Convert main start and end dates
facebook_ads['Start_Date'] = pd.to_datetime(facebook_ads['Starts'], errors='coerce')
facebook_ads['End_Date'] = pd.to_datetime(facebook_ads['Ends'], errors='coerce')
# Clean and convert the extracted 'DATE' column (removing accidental appended strings)
if 'DATE' in facebook_ads.columns:
    facebook_ads['DATE'] = facebook_ads['DATE'].astype(str).str.replace(' - Copy', '', regex=False)
    facebook_ads['DATE'] = pd.to_datetime(facebook_ads['DATE'], errors='coerce')
# We use drop() with axis=1 (or columns=[...]) to remove them entirely
facebook_ads = facebook_ads.drop(columns=['Starts', 'Ends'], errors='ignore')

C:\Users\foogu\AppData\Local\Temp\ipykernel_3492\1730411335.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  facebook_ads['Start_Date'] = pd.to_datetime(facebook_ads['Starts'], errors='coerce')
C:\Users\foogu\AppData\Local\Temp\ipykernel_3492\1730411335.py:4: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  facebook_ads['End_Date'] = pd.to_datetime(facebook_ads['Ends'], errors='coerce')


In [9]:
# 1. Define the reference date for active campaigns (e.g., today)
# Alternatively, if you know the exact date you pulled this data, use that (e.g., pd.to_datetime('2025-12-31'))
reference_date = pd.Timestamp.now() 

# 2. Calculate duration by temporarily filling NaT with the reference date
facebook_ads['Campaign Duration (Days)'] = (facebook_ads['End_Date'].fillna(reference_date) - facebook_ads['Start_Date']).dt.days
# Extract Year-Month
facebook_ads['Year_Month'] = facebook_ads['Start_Date'].dt.to_period('M')

In [10]:
facebook_ads.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3421 entries, 0 to 3986
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Ad name                   3421 non-null   object        
 1   Delivery status           3421 non-null   object        
 2   Delivery level            3421 non-null   object        
 3   Reach                     3421 non-null   int64         
 4   Impressions               3421 non-null   int64         
 5   Frequency                 3421 non-null   float64       
 6   Result type               2793 non-null   object        
 7   Results                   2784 non-null   object        
 8   Amount spent (THB)        3421 non-null   float64       
 9   Cost per result           2793 non-null   float64       
 10  Restaurant ID             2866 non-null   object        
 11  Rest 2                    43 non-null     float64       
 12  Rest 3                   

In [11]:
# 1. Define the list of outlier Restaurant IDs
outlier_ids = ['4503', '4502', '933', '837']

# 2. Join them into a single regular expression pattern separated by the OR operator (|)
# This creates the pattern: '4503|4502|933|837'
pattern = '|'.join(outlier_ids)

# 3. Filter the dataframe
# We use the tilde (~) to say "KEEP the rows that DO NOT contain this pattern"
# (Assuming you renamed the column to 'Restaurant ID' as discussed previously)
facebook_ads = facebook_ads[~facebook_ads['Restaurant ID'].astype(str).str.contains(pattern, na=False)]
facebook_ads.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2993 entries, 31 to 3986
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Ad name                   2993 non-null   object        
 1   Delivery status           2993 non-null   object        
 2   Delivery level            2993 non-null   object        
 3   Reach                     2993 non-null   int64         
 4   Impressions               2993 non-null   int64         
 5   Frequency                 2993 non-null   float64       
 6   Result type               2420 non-null   object        
 7   Results                   2411 non-null   object        
 8   Amount spent (THB)        2993 non-null   float64       
 9   Cost per result           2420 non-null   float64       
 10  Restaurant ID             2438 non-null   object        
 11  Rest 2                    12 non-null     float64       
 12  Rest 3                  

In [12]:
facebook_ads['Campaign'].value_counts()

Campaign
res                                  1214
album_res                             388
album_res_grp                         152
album                                  78
MP - OCT'25                            15
TOURIST-ASIA-COUNTRY                   13
MP - NOV'25                            11
Happy Lunar New Year 2025              10
TOURIST-KOREA                           8
TOURIST-WESTERN                         7
MP - SEP'25                             7
single                                  6
TOURIST-SEA-COUNTRY                     6
TOURIST-MENA-COUNTRY                    5
res_grp                                 4
FESTIVE MP-DEC'25                       3
HH Super Brand Day 2025                 3
TOURIST-JAPAN                           3
Time to Rejoin in 2025 ad - Copy        3
57th Street Marriott - Copy             3
Songkran Splash Deals 2025              2
Songkran Splash Deals 2025 - Copy       2
res_res                                 2
Valentine Day 2025 - Copy

In [13]:
#To fill up the Results column with 0 for the rows where it's NaN, and derive Cost Per Result by dividing Amount Spent (THB) with Results

facebook_ads['Results'] = pd.to_numeric(facebook_ads['Results'], errors='coerce').fillna(0).astype(int)

# Calculate cost per result, avoiding division by zero
facebook_ads['Cost per result'] = np.where(
    facebook_ads['Results'] > 0,
    facebook_ads['Amount spent (THB)'] / facebook_ads['Results'],
    0
)

In [14]:
facebook_ads.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2993 entries, 31 to 3986
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Ad name                   2993 non-null   object        
 1   Delivery status           2993 non-null   object        
 2   Delivery level            2993 non-null   object        
 3   Reach                     2993 non-null   int64         
 4   Impressions               2993 non-null   int64         
 5   Frequency                 2993 non-null   float64       
 6   Result type               2420 non-null   object        
 7   Results                   2993 non-null   int32         
 8   Amount spent (THB)        2993 non-null   float64       
 9   Cost per result           2993 non-null   float64       
 10  Restaurant ID             2438 non-null   object        
 11  Rest 2                    12 non-null     float64       
 12  Rest 3                  

In [15]:
#Efficiency metrics calculation for Facebook Ads
# 1. Conversion Rate (Results / Impressions) - shows % of impressions that converted
facebook_ads['Conversion_Rate_Pct'] = np.where(
    facebook_ads['Impressions'] > 0,
    (facebook_ads['Results'] / facebook_ads['Impressions']) * 100,
    0
)

# 2. CTR Proxy (Results / Reach) - assumes results require a click
facebook_ads['CTR_Proxy_Pct'] = np.where(
    facebook_ads['Reach'] > 0,
    (facebook_ads['Results'] / facebook_ads['Reach']) * 100,
    0
)

# 3. CPM (Cost Per Mille/Thousand Impressions)
facebook_ads['CPM'] = np.where(
    facebook_ads['Impressions'] > 0,
    (facebook_ads['Amount spent (THB)'] / facebook_ads['Impressions']) * 1000,
    0
)

# 4. Cost Per Reach
facebook_ads['Cost_per_Reach'] = np.where(
    facebook_ads['Reach'] > 0,
    facebook_ads['Amount spent (THB)'] / facebook_ads['Reach'],
    0
)

# We grey these segments because currently unclear on the campaign duration until further clarity achieved. 
# # 5. Daily Spend (for campaigns with duration)
# facebook_ads['Daily_Spend'] = np.where(
#     facebook_ads['Campaign Duration (Days)'] > 0,
#     facebook_ads['Amount spent (THB)'] / facebook_ads['Campaign Duration (Days)'],
#     0
# )

# # 6. Results per Day
# facebook_ads['Results_per_Day'] = np.where(
#     facebook_ads['Campaign Duration (Days)'] > 0,
#     facebook_ads['Results'] / facebook_ads['Campaign Duration (Days)'],
#     0
# )

In [16]:
# ============================================================================
# AGGREGATE TO RESTAURANT-MONTH LEVEL
# ============================================================================
print("\n" + "="*80)
print("STEP 7: AGGREGATING TO RESTAURANT-MONTH LEVEL")
print("="*80)

# Filter valid data
fb_ads_valid = facebook_ads[
    (facebook_ads['Restaurant ID'].notna()) & 
    (facebook_ads['Restaurant ID'] != "~") &
    (facebook_ads['Year_Month'].notna())
].copy()

print(f"Valid rows for aggregation: {len(fb_ads_valid)}")

# Aggregate
restaurant_month_agg = fb_ads_valid.groupby(['Restaurant ID', 'Year_Month']).agg({
    'Ad name': 'count',
    'Amount spent (THB)': 'sum',
    'Results': 'sum',
    'Reach': 'sum',
    'Impressions': 'sum',
    'Frequency': 'mean',
    'Cost per result': lambda x: x[x > 0].mean() if len(x[x > 0]) > 0 else 0,
    'Conversion_Rate_Pct': 'mean',
    'CTR_Proxy_Pct': 'mean',
    'CPM': 'mean',
    'Cost_per_Reach': 'mean',
    # 'Campaign_Source': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'UNKNOWN',
    # 'Campaign_Type': lambda x: ', '.join(x.unique()[:3]),
    # 'Has_KOL': 'sum',
    'Campaign Duration (Days)': 'mean',
    # 'Daily_Spend': 'mean',
    # 'Results_per_Day': 'mean',
}).reset_index()


STEP 7: AGGREGATING TO RESTAURANT-MONTH LEVEL
Valid rows for aggregation: 2438


In [17]:
# Rename columns
restaurant_month_agg.columns = [
    'Restaurant_ID', 'Year_Month', 'Num_Ads', 'Total_Spend_THB', 'Total_Results',
    'Total_Reach', 'Total_Impressions', 'Avg_Frequency',
    'Avg_Cost_per_Result', 'Avg_Conversion_Rate_Pct', 'Avg_CTR_Proxy_Pct',
    'Avg_CPM', 'Avg_Cost_per_Reach', 'Avg_Campaign_Duration_Days',
]

# Calculate overall metrics
restaurant_month_agg['Overall_Conversion_Rate_Pct'] = np.where(
    restaurant_month_agg['Total_Impressions'] > 0,
    (restaurant_month_agg['Total_Results'] / restaurant_month_agg['Total_Impressions']) * 100,
    0
)

restaurant_month_agg['Overall_Cost_per_Result'] = np.where(
    restaurant_month_agg['Total_Results'] > 0,
    restaurant_month_agg['Total_Spend_THB'] / restaurant_month_agg['Total_Results'],
    0
)

restaurant_month_agg['Results_per_1000_THB'] = np.where(
    restaurant_month_agg['Total_Spend_THB'] > 0,
    (restaurant_month_agg['Total_Results'] / restaurant_month_agg['Total_Spend_THB']) * 1000,
    0
)

print(f"\n✓ Aggregation complete")
print(f"Total restaurant-month combinations: {len(restaurant_month_agg)}")
print(f"Unique restaurants: {restaurant_month_agg['Restaurant_ID'].nunique()}")


✓ Aggregation complete
Total restaurant-month combinations: 1010
Unique restaurants: 456


In [18]:
# ============================================================================
# SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("STEP 8: SUMMARY STATISTICS")
print("="*80)

print("\n--- OVERALL PERFORMANCE ---")
print(f"Total spend: {restaurant_month_agg['Total_Spend_THB'].sum():,.2f} THB")
print(f"Total conversions: {restaurant_month_agg['Total_Results'].sum():,}")
print(f"Total reach: {restaurant_month_agg['Total_Reach'].sum():,}")
print(f"Total impressions: {restaurant_month_agg['Total_Impressions'].sum():,}")

print("\n--- FORMAT DISTRIBUTION ---")
print(facebook_ads['Format_Type'].value_counts())

print("\n--- DATA BY FORMAT ---")
format_summary = facebook_ads.groupby('Format_Type').agg({
    'Amount spent (THB)': 'sum',
    'Results': 'sum',
    'Restaurant ID': lambda x: x.notna().sum()
}).round(2)
format_summary.columns = ['Total_Spend_THB', 'Total_Results', 'Restaurants_with_ID']
print(format_summary)


STEP 8: SUMMARY STATISTICS

--- OVERALL PERFORMANCE ---
Total spend: 4,405,590.39 THB
Total conversions: 181,765
Total reach: 62,087,513
Total impressions: 131,780,941

--- FORMAT DISTRIBUTION ---
Format_Type
LEGACY        2620
STRUCTURED     335
SIMPLE          38
Name: count, dtype: int64

--- DATA BY FORMAT ---
             Total_Spend_THB  Total_Results  Restaurants_with_ID
Format_Type                                                     
LEGACY            4451351.67         193553                 2132
SIMPLE              30791.24           1259                   38
STRUCTURED         445635.85          16700                  268


In [19]:
restaurant_month_agg.head()

,Restaurant_ID,Year_Month,Num_Ads,Total_Spend_THB,Total_Results,Total_Reach,Total_Impressions,Avg_Frequency,Avg_Cost_per_Result,Avg_Conversion_Rate_Pct,Avg_CTR_Proxy_Pct,Avg_CPM,Avg_Cost_per_Reach,Avg_Campaign_Duration_Days,Overall_Conversion_Rate_Pct,Overall_Cost_per_Result,Results_per_1000_THB
0,10,2025-04,1,156.69,36,2113,2924,1.383814,4.352500,1.231190,1.703739,53.587551,0.074155,298.000000,1.231190,4.352500,229.753016
1,1003,2024-08,1,1009.45,44,16308,35606,2.183346,22.942045,0.123575,0.269806,28.350559,0.061899,540.000000,0.123575,22.942045,43.588093
2,1003,2024-09,3,86.10,3,1307,1846,1.220532,26.756667,0.060314,0.086957,34.238692,0.043698,519.666667,0.162514,28.700000,34.843206
3,1003,2024-11,2,926.44,102,8929,19233,1.913007,15.522347,0.402454,0.817620,51.741248,0.097648,450.500000,0.530338,9.082745,110.098873
4,1003,2025-03,4,68.41,1,1770,2058,1.143484,33.970000,0.022978,0.026596,33.955116,0.038835,244.750000,0.048591,68.410000,14.617746
